In [1]:
# Zelle 1: Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import io
import base64
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score
import pymongo
import os
from tensorflow.keras.utils import Sequence, to_categorical
import gc  # Garbage Collector


2024-11-08 13:42:03.835436: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-11-08 13:42:03.858041: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-11-08 13:42:04.180883: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
# Zelle 2: Datenbankverbindung und Datenladung
def load_data():
    mongo_uri = os.getenv('MONGO_URI', 'mongodb://localhost:27017/fingerprintDB')
    client = pymongo.MongoClient(mongo_uri)
    db = client.get_default_database()
    
    # Daten laden
    canvassamples = list(db['canvassamples'].find())
    fingerprints = list(db['fingerprints'].find())
    
    # In DataFrames konvertieren
    canvassamples_df = pd.DataFrame(canvassamples)
    fingerprints_df = pd.DataFrame(fingerprints)
    
    # Merge der DataFrames
    merged_df = pd.merge(canvassamples_df, fingerprints_df, 
                        left_on='fingerprintId', 
                        right_on='_id', 
                        suffixes=('_sample', '_fingerprint'))
    
    return merged_df

# Daten laden
merged_df = load_data()
print(f"Gesamtdatensatz enthält {len(merged_df)} Einträge.")

Gesamtdatensatz enthält 2000000 Einträge.


In [ ]:
# Benutzer-IDs extrahieren
user_ids = merged_df['username'].unique()

# DataFrames für jeden Benutzer erstellen
user_dfs = {user_id: merged_df[merged_df['username'] == user_id] for user_id in user_ids}

# Beispielhafte Anzeige der ersten 5 Zeilen für jeden Benutzer
for user_id, df in user_dfs.items():
    print(f"Erste 5 Zeilen für Benutzer {user_id}:")
    display(df.head())

Erste 5 Zeilen für Benutzer benutzername_1:


,_id_sample,fingerprintId,sampleData,createdAt_sample,__v_sample,_id_fingerprint,fingerprint,fingerprintHash,username,name,deviceName,operatingSystem,canvasSampleCount,createdAt_fingerprint,__v_fingerprint
0,66e326e9fab6918841db55df,66e326e8fab6918841db55d1,iVBORw0KGgoAAAANSUhEUgAAARgAAAAjCAYAAABPRBVWAA...,2024-09-12 17:37:45.906,0,66e326e8fab6918841db55d1,"{""UserAgent"":""Mozilla/5.0 (Windows NT 10.0; Wi...",609903dc45201305b1c661643326d37bacd216f3c4fe13...,benutzername_1,Benutzer 1,ThinkPad E14 Gen 5,Windows 11,100000,2024-09-12 17:37:44.458,0
1,66e326e9fab6918841db55db,66e326e8fab6918841db55d1,iVBORw0KGgoAAAANSUhEUgAAARgAAAAjCAYAAABPRBVWAA...,2024-09-12 17:37:45.847,0,66e326e8fab6918841db55d1,"{""UserAgent"":""Mozilla/5.0 (Windows NT 10.0; Wi...",609903dc45201305b1c661643326d37bacd216f3c4fe13...,benutzername_1,Benutzer 1,ThinkPad E14 Gen 5,Windows 11,100000,2024-09-12 17:37:44.458,0
2,66e326e9fab6918841db55d6,66e326e8fab6918841db55d1,iVBORw0KGgoAAAANSUhEUgAAARgAAAAjCAYAAABPRBVWAA...,2024-09-12 17:37:45.772,0,66e326e8fab6918841db55d1,"{""UserAgent"":""Mozilla/5.0 (Windows NT 10.0; Wi...",609903dc45201305b1c661643326d37bacd216f3c4fe13...,benutzername_1,Benutzer 1,ThinkPad E14 Gen 5,Windows 11,100000,2024-09-12 17:37:44.458,0
3,66e326e9fab6918841db55e3,66e326e8fab6918841db55d1,iVBORw0KGgoAAAANSUhEUgAAARgAAAAjCAYAAABPRBVWAA...,2024-09-12 17:37:45.981,0,66e326e8fab6918841db55d1,"{""UserAgent"":""Mozilla/5.0 (Windows NT 10.0; Wi...",609903dc45201305b1c661643326d37bacd216f3c4fe13...,benutzername_1,Benutzer 1,ThinkPad E14 Gen 5,Windows 11,100000,2024-09-12 17:37:44.458,0
4,66e326eafab6918841db55eb,66e326e8fab6918841db55d1,iVBORw0KGgoAAAANSUhEUgAAARgAAAAjCAYAAABPRBVWAA...,2024-09-12 17:37:46.118,0,66e326e8fab6918841db55d1,"{""UserAgent"":""Mozilla/5.0 (Windows NT 10.0; Wi...",609903dc45201305b1c661643326d37bacd216f3c4fe13...,benutzername_1,Benutzer 1,ThinkPad E14 Gen 5,Windows 11,100000,2024-09-12 17:37:44.458,0


Erste 5 Zeilen für Benutzer benutzername_3:


,_id_sample,fingerprintId,sampleData,createdAt_sample,__v_sample,_id_fingerprint,fingerprint,fingerprintHash,username,name,deviceName,operatingSystem,canvasSampleCount,createdAt_fingerprint,__v_fingerprint
100000,6703f8369717b7a0d179d6e3,6703f8339717b7a0d179d6df,iVBORw0KGgoAAAANSUhEUgAAARgAAAAjCAYAAABPRBVWAA...,2024-10-07 15:03:18.318,0,6703f8339717b7a0d179d6df,"{""UserAgent"":""Mozilla/5.0 (iPhone; CPU iPhone ...",0bfc7806ac7a11fd13a1a7cb8a309468179bf64f711eb3...,benutzername_3,Benutzer 3,iPhone 11,IOS 17.5.1,100000,2024-10-07 15:03:15.273,0
100001,6703f8369717b7a0d179d6e7,6703f8339717b7a0d179d6df,iVBORw0KGgoAAAANSUhEUgAAARgAAAAjCAYAAABPRBVWAA...,2024-10-07 15:03:18.339,0,6703f8339717b7a0d179d6df,"{""UserAgent"":""Mozilla/5.0 (iPhone; CPU iPhone ...",0bfc7806ac7a11fd13a1a7cb8a309468179bf64f711eb3...,benutzername_3,Benutzer 3,iPhone 11,IOS 17.5.1,100000,2024-10-07 15:03:15.273,0
100002,6703f8369717b7a0d179d6eb,6703f8339717b7a0d179d6df,iVBORw0KGgoAAAANSUhEUgAAARgAAAAjCAYAAABPRBVWAA...,2024-10-07 15:03:18.359,0,6703f8339717b7a0d179d6df,"{""UserAgent"":""Mozilla/5.0 (iPhone; CPU iPhone ...",0bfc7806ac7a11fd13a1a7cb8a309468179bf64f711eb3...,benutzername_3,Benutzer 3,iPhone 11,IOS 17.5.1,100000,2024-10-07 15:03:15.273,0
100003,6703f8369717b7a0d179d6ef,6703f8339717b7a0d179d6df,iVBORw0KGgoAAAANSUhEUgAAARgAAAAjCAYAAABPRBVWAA...,2024-10-07 15:03:18.379,0,6703f8339717b7a0d179d6df,"{""UserAgent"":""Mozilla/5.0 (iPhone; CPU iPhone ...",0bfc7806ac7a11fd13a1a7cb8a309468179bf64f711eb3...,benutzername_3,Benutzer 3,iPhone 11,IOS 17.5.1,100000,2024-10-07 15:03:15.273,0
100004,6703f8369717b7a0d179d6f7,6703f8339717b7a0d179d6df,iVBORw0KGgoAAAANSUhEUgAAARgAAAAjCAYAAABPRBVWAA...,2024-10-07 15:03:18.412,0,6703f8339717b7a0d179d6df,"{""UserAgent"":""Mozilla/5.0 (iPhone; CPU iPhone ...",0bfc7806ac7a11fd13a1a7cb8a309468179bf64f711eb3...,benutzername_3,Benutzer 3,iPhone 11,IOS 17.5.1,100000,2024-10-07 15:03:15.273,0


Erste 5 Zeilen für Benutzer benutzername_4:


...

Erste 5 Zeilen für Benutzer benutzername_21:


,_id_sample,fingerprintId,sampleData,createdAt_sample,__v_sample,_id_fingerprint,fingerprint,fingerprintHash,username,name,deviceName,operatingSystem,canvasSampleCount,createdAt_fingerprint,__v_fingerprint
1800000,6707c487ffc8085dd625a040,6707c486ffc8085dd625a03b,iVBORw0KGgoAAAANSUhEUgAAARgAAAAjCAYAAABPRBVWAA...,2024-10-10 12:11:51.494,0,6707c486ffc8085dd625a03b,"{""UserAgent"":""Mozilla/5.0 (Macintosh; Intel Ma...",ceb06c001f5b9b3b7a6ac3c7e77def777ae8b9c977a4f6...,benutzername_21,Benutzer 21,MacMini M1 2020,Ventura 13.2,100000,2024-10-10 12:11:50.062,0
1800001,6707c487ffc8085dd625a044,6707c486ffc8085dd625a03b,iVBORw0KGgoAAAANSUhEUgAAARgAAAAjCAYAAABPRBVWAA...,2024-10-10 12:11:51.506,0,6707c486ffc8085dd625a03b,"{""UserAgent"":""Mozilla/5.0 (Macintosh; Intel Ma...",ceb06c001f5b9b3b7a6ac3c7e77def777ae8b9c977a4f6...,benutzername_21,Benutzer 21,MacMini M1 2020,Ventura 13.2,100000,2024-10-10 12:11:50.062,0
1800002,6707c487ffc8085dd625a048,6707c486ffc8085dd625a03b,iVBORw0KGgoAAAANSUhEUgAAARgAAAAjCAYAAABPRBVWAA...,2024-10-10 12:11:51.517,0,6707c486ffc8085dd625a03b,"{""UserAgent"":""Mozilla/5.0 (Macintosh; Intel Ma...",ceb06c001f5b9b3b7a6ac3c7e77def777ae8b9c977a4f6...,benutzername_21,Benutzer 21,MacMini M1 2020,Ventura 13.2,100000,2024-10-10 12:11:50.062,0
1800003,6707c487ffc8085dd625a04c,6707c486ffc8085dd625a03b,iVBORw0KGgoAAAANSUhEUgAAARgAAAAjCAYAAABPRBVWAA...,2024-10-10 12:11:51.528,0,6707c486ffc8085dd625a03b,"{""UserAgent"":""Mozilla/5.0 (Macintosh; Intel Ma...",ceb06c001f5b9b3b7a6ac3c7e77def777ae8b9c977a4f6...,benutzername_21,Benutzer 21,MacMini M1 2020,Ventura 13.2,100000,2024-10-10 12:11:50.062,0
1800004,6707c487ffc8085dd625a050,6707c486ffc8085dd625a03b,iVBORw0KGgoAAAANSUhEUgAAARgAAAAjCAYAAABPRBVWAA...,2024-10-10 12:11:51.539,0,6707c486ffc8085dd625a03b,"{""UserAgent"":""Mozilla/5.0 (Macintosh; Intel Ma...",ceb06c001f5b9b3b7a6ac3c7e77def777ae8b9c977a4f6...,benutzername_21,Benutzer 21,MacMini M1 2020,Ventura 13.2,100000,2024-10-10 12:11:50.062,0


Erste 5 Zeilen für Benutzer benutzername_22:


,_id_sample,fingerprintId,sampleData,createdAt_sample,__v_sample,_id_fingerprint,fingerprint,fingerprintHash,username,name,deviceName,operatingSystem,canvasSampleCount,createdAt_fingerprint,__v_fingerprint
1900000,6707cabdffc8085dd62a72c5,6707cabaffc8085dd62a6eb2,iVBORw0KGgoAAAANSUhEUgAAARgAAAAjCAYAAABPRBVWAA...,2024-10-10 12:38:21.957,0,6707cabaffc8085dd62a6eb2,"{""UserAgent"":""Mozilla/5.0 (Macintosh; Intel Ma...",7c57994e6cde8672f73c095d7d838960776641b9ac240b...,benutzername_22,Benutzer 22,M2 mac mini,macOS Sequoia 15.0,100000,2024-10-10 12:38:18.801,0
1900001,6707cabdffc8085dd62a72d1,6707cabaffc8085dd62a6eb2,iVBORw0KGgoAAAANSUhEUgAAARgAAAAjCAYAAABPRBVWAA...,2024-10-10 12:38:21.970,0,6707cabaffc8085dd62a6eb2,"{""UserAgent"":""Mozilla/5.0 (Macintosh; Intel Ma...",7c57994e6cde8672f73c095d7d838960776641b9ac240b...,benutzername_22,Benutzer 22,M2 mac mini,macOS Sequoia 15.0,100000,2024-10-10 12:38:18.801,0
1900002,6707cabdffc8085dd62a72ca,6707cabaffc8085dd62a6eb2,iVBORw0KGgoAAAANSUhEUgAAARgAAAAjCAYAAABPRBVWAA...,2024-10-10 12:38:21.964,0,6707cabaffc8085dd62a6eb2,"{""UserAgent"":""Mozilla/5.0 (Macintosh; Intel Ma...",7c57994e6cde8672f73c095d7d838960776641b9ac240b...,benutzername_22,Benutzer 22,M2 mac mini,macOS Sequoia 15.0,100000,2024-10-10 12:38:18.801,0
1900003,6707cabdffc8085dd62a72dd,6707cabaffc8085dd62a6eb2,iVBORw0KGgoAAAANSUhEUgAAARgAAAAjCAYAAABPRBVWAA...,2024-10-10 12:38:21.981,0,6707cabaffc8085dd62a6eb2,"{""UserAgent"":""Mozilla/5.0 (Macintosh; Intel Ma...",7c57994e6cde8672f73c095d7d838960776641b9ac240b...,benutzername_22,Benutzer 22,M2 mac mini,macOS Sequoia 15.0,100000,2024-10-10 12:38:18.801,0
1900004,6707cabdffc8085dd62a72e1,6707cabaffc8085dd62a6eb2,iVBORw0KGgoAAAANSUhEUgAAARgAAAAjCAYAAABPRBVWAA...,2024-10-10 12:38:21.985,0,6707cabaffc8085dd62a6eb2,"{""UserAgent"":""Mozilla/5.0 (Macintosh; Intel Ma...",7c57994e6cde8672f73c095d7d838960776641b9ac240b...,benutzername_22,Benutzer 22,M2 mac mini,macOS Sequoia 15.0,100000,2024-10-10 12:38:18.801,0


: 

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import io
import base64
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.applications import ResNet50, MobileNetV2, EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
import gc  # Garbage Collector

# Funktion zur Verarbeitung der Bilder in Graustufen
def process_image_grayscale(base64_str, target_size=(224, 224)):
    try:
        image_data = base64.b64decode(base64_str)
        image = Image.open(io.BytesIO(image_data))
        image = image.convert('L')  # Konvertieren in Graustufen
        image = image.resize(target_size)
        image_array = np.array(image) / 255.0  # Normalisierung auf Werte zwischen 0 und 1
        image_array = np.stack((image_array,)*3, axis=-1)  # Konvertieren in 3-Kanal-Bild
        return image_array
    except Exception as e:
        print(f"Fehler bei der Bildverarbeitung: {e}")
        return None

def extract_images_from_df(df):
    images = []
    labels = []
    for _, row in df.iterrows():
        image = process_image_grayscale(row['sampleData'])
        if image is not None:
            images.append(image)
            labels.append(1 if row['username'] == example_user_id else 0)
    return np.array(images), np.array(labels)

# Benutzer-IDs extrahieren
user_ids = merged_df['username'].unique()

# DataFrames für jeden Benutzer erstellen
user_dfs = {user_id: merged_df[merged_df['username'] == user_id] for user_id in user_ids}

# Beispiel für Benutzer 1
example_user_id = 'benutzername_1'
user_df = user_dfs[example_user_id]

# Negative Beispiele hinzufügen und begrenzen
negative_df = pd.concat([user_dfs[user_id] for user_id in user_ids if user_id != example_user_id])
negative_df = negative_df.sample(n=100000, random_state=42)  # Zufällige Stichprobe von 100.000 negativen Beispielen

# Aufteilung in Train/Val/Test-Sets (70% Train, 20% Val, 10% Test)
train_df, test_df = train_test_split(pd.concat([user_df, negative_df]), test_size=0.1, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.2222, random_state=42)  # 0.2222 * 0.9 ≈ 0.2

# Bilddaten verarbeiten
X_train, y_train = extract_images_from_df(train_df)
X_val, y_val = extract_images_from_df(val_df)
X_test, y_test = extract_images_from_df(test_df)

# One-Hot-Encoding der Labels
y_train = to_categorical(y_train, num_classes=2)
y_val = to_categorical(y_val, num_classes=2)
y_test = to_categorical(y_test, num_classes=2)

# Verwendung eines vortrainierten ResNet-Modells
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(2, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

for layer in base_model.layers:
    layer.trainable = False

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Modell trainieren
history = model.fit(X_train, y_train, epochs=50, validation_data=(X_val, y_val))

# Zufällige Samples von verschiedenen Benutzern auswählen
random_samples = []
random_labels = []
sample_users = []

# 2 Samples von Benutzer 1
user_df = user_dfs[example_user_id]
samples = user_df.sample(n=2, random_state=42)
random_samples.extend(samples['sampleData'].tolist())
sample_users.extend([example_user_id] * 2)
random_labels.extend([1] * 2)  # Label 1 für Benutzer 1

# 8 Samples von anderen Benutzern
other_user_ids = [user_id for user_id in user_ids if user_id != example_user_id]
for user_id in np.random.choice(other_user_ids, 8, replace=False):
    user_df = user_dfs[user_id]
    sample = user_df.sample(n=1, random_state=42)
    random_samples.extend(sample['sampleData'].tolist())
    sample_users.extend([user_id])
    random_labels.extend([0])  # Label 0 für andere Benutzer

# Bilddaten verarbeiten und Features extrahieren
X_random = np.array([process_image_grayscale(sample) for sample in random_samples])

# Vorhersagen mit dem Modell durchführen
y_pred = model.predict(X_random)
y_pred_classes = np.argmax(y_pred, axis=1)
y_pred_prob = y_pred[:, 1]  # Wahrscheinlichkeiten für Klasse 1 (Benutzer 1)

# Wahrscheinlichkeiten und Labels in einem DataFrame speichern
results_df = pd.DataFrame({
    'User': sample_users,
    'True Label': random_labels,
    'Predicted Label': y_pred_classes
})

# Ob die Vorhersage korrekt ist
results_df['Correct'] = results_df['True Label'] == results_df['Predicted Label']

# Klassifikationstyp hinzufügen
def get_classification(row):
    if row['True Label'] == 1 and row['Predicted Label'] == 1:
        return 'True Positive'
    elif row['True Label'] == 0 and row['Predicted Label'] == 1:
        return 'False Positive'
    elif row['True Label'] == 0 and row['Predicted Label'] == 0:
        return 'True Negative'
    else:
        return 'False Negative'

results_df['Classification'] = results_df.apply(get_classification, axis=1)

# Tabelle anzeigen
display(results_df)

# Farben für True Positives, False Positives, True Negatives und False Negatives
def get_color(row):
    if row['True Label'] == 1 and row['Predicted Label'] == 1:
        return 'green'  # True Positive
    elif row['True Label'] == 0 and row['Predicted Label'] == 1:
        return 'red'    # False Positive
    elif row['True Label'] == 0 and row['Predicted Label'] == 0:
        return 'blue'   # True Negative
    else:
        return 'orange' # False Negative

results_df['Color'] = results_df.apply(get_color, axis=1)

# Graphen erstellen
plt.figure(figsize=(12, 8))
plt.scatter(range(len(results_df)), results_df['Predicted Label'], c=results_df['Color'], alpha=0.5)
plt.title('Klassifikationen für Benutzer 1')
plt.xlabel('Sample Index')
plt.ylabel('Vorhergesagte Klasse')
plt.yticks([0, 1], ['Nicht Benutzer 1', 'Benutzer 1'])
plt.show()

: 

: 